In [6]:
import time
import json
import requests

class APIwrapper:
    _access_point = "https://api.ukhsa-dashboard.data.gov.uk"
    _last_access = 0.0
    
    def __init__(self, theme, sub_theme, topic, geography_type, geography, metric):
        url_path = (f"/themes/{theme}/sub_themes/{sub_theme}/topics/{topic}/geography_types/" +
                  f"{geography_type}/geographies/{geography}/metrics/{metric}")
        self._start_url = APIwrapper._access_point + url_path
        self._filters = None
        self._page_size = -1
        self.count = None

    def get_page(self, filters={}, page_size=365):
        if page_size > 365:
            raise ValueError("Max supported page size is 365")
        if filters != self._filters or page_size != self._page_size:
            self._filters = filters
            self._page_size = page_size
            self._next_url = self._start_url
        if self._next_url is None: 
            return [] 
        
        curr_time = time.time()
        deltat = curr_time - APIwrapper._last_access
        if deltat < 0.33: 
            time.sleep(0.33 - deltat)
        APIwrapper._last_access = time.time()
        
        parameters = {x: y for x, y in filters.items() if y is not None}
        parameters['page_size'] = page_size
        
        response = requests.get(self._next_url, params=parameters)
        
        # Check if request was successful
        if response.status_code != 200:
            print(f"API Error: Status code {response.status_code}")
            print(response.text)
            return []
            
        res_json = response.json()
        self._next_url = res_json.get('next')
        self.count = res_json.get('count', 0)
        return res_json.get('results', []) 

    def get_all_pages(self, filters={}, page_size=365):
        data = []
        while True:
            next_page = self.get_page(filters, page_size)
            if not next_page:
                break 
            data.extend(next_page)
        return data

if __name__ == "__main__":
    # Updated with correct API path convention for RSV
     rsv_api = APIwrapper(
         theme="infectious_disease",
         sub_theme="respiratory",
         topic="respiratory-syncytial-virus-rsv",
         geography_type="Nation",
         geography="England",
         metric="respiratory-syncytial-virus-rsv_casesByDay"
    )

    print("Fetching all pages from UKHSA API...")
    all_rsv_data = rsv_api.get_all_pages()
    print(f"Successfully fetched {len(all_rsv_data)} items (Total count reported: {rsv_api.count}).")

    if len(all_rsv_data) > 0:
        output_filename = "ukhsa-Weekly_RSV_data.json"
        with open(output_filename, "w") as f:
            json.dump(all_rsv_data, f, indent=4)
        print(f"Data successfully saved to {output_filename}")
    else:
        print("Warning: 0 items returned. Please check your metric spelling or verify with your local JSON headers.")

IndentationError: unindent does not match any outer indentation level (<string>, line 69)

In [1]:
import requests

import pandas as pd

url = (

    "https://api.ukhsa-dashboard.data.gov.uk"

    "/themes/infectious_disease"

    "/sub_themes/respiratory"

    "/topics/RSV"

    "/geography_types/Nation"

    "/geographies/England"

    "/metrics/RSV_testing_positivityByWeek"

)

all_results = []

next_url = url

while next_url:

    response = requests.get(

        next_url,

        params={"page_size": 100} if next_url == url else None,

        timeout=30

    )

    response.raise_for_status()

    data = response.json()

    all_results.extend(data["results"])

    next_url = data.get("next")

print("Total downloaded records:", len(all_results))

Total downloaded records: 3532


In [2]:
df = pd.DataFrame(all_results)

print("Age groups:", df["age"].unique())
print("Sex categories:", df["sex"].unique())
print("Strata:", df["stratum"].unique())

display(df.head())

rsv_df = df[
    (df["age"] == "all") &
    (df["sex"] == "all") &
    (df["stratum"] == "default")
].copy()

print("Filtered records:", len(rsv_df))

display(rsv_df.head())

Age groups: ['45-64' '80+' '00-04' '65-79' '05-14' 'all' '15-44' '65+' '65']
Sex categories: ['all']
Strata: ['default']


,theme,sub_theme,topic,geography_type,geography,geography_code,metric,metric_group,stratum,sex,age,year,month,epiweek,date,metric_value,in_reporting_delay_period
0,infectious_disease,respiratory,RSV,Nation,England,E92000001,RSV_testing_positivityByWeek,testing,default,all,45-64,2017,7,27,2017-07-03,0.00,False
1,infectious_disease,respiratory,RSV,Nation,England,E92000001,RSV_testing_positivityByWeek,testing,default,all,80+,2017,7,27,2017-07-03,0.00,False
2,infectious_disease,respiratory,RSV,Nation,England,E92000001,RSV_testing_positivityByWeek,testing,default,all,00-04,2017,7,27,2017-07-03,0.69,False
3,infectious_disease,respiratory,RSV,Nation,England,E92000001,RSV_testing_positivityByWeek,testing,default,all,65-79,2017,7,27,2017-07-03,0.59,False
4,infectious_disease,respiratory,RSV,Nation,England,E92000001,RSV_testing_positivityByWeek,testing,default,all,05-14,2017,7,27,2017-07-03,0.00,False


Filtered records: 483


,theme,sub_theme,topic,geography_type,geography,geography_code,metric,metric_group,stratum,sex,age,year,month,epiweek,date,metric_value,in_reporting_delay_period
5,infectious_disease,respiratory,RSV,Nation,England,E92000001,RSV_testing_positivityByWeek,testing,default,all,all,2017,7,27,2017-07-03,0.29,False
7,infectious_disease,respiratory,RSV,Nation,England,E92000001,RSV_testing_positivityByWeek,testing,default,all,all,2017,7,28,2017-07-10,0.46,False
20,infectious_disease,respiratory,RSV,Nation,England,E92000001,RSV_testing_positivityByWeek,testing,default,all,all,2017,7,29,2017-07-17,0.40,False
26,infectious_disease,respiratory,RSV,Nation,England,E92000001,RSV_testing_positivityByWeek,testing,default,all,all,2017,7,30,2017-07-24,0.81,False
34,infectious_disease,respiratory,RSV,Nation,England,E92000001,RSV_testing_positivityByWeek,testing,default,all,all,2017,7,31,2017-07-31,1.17,False


In [3]:
from pathlib import Path

rsv_clean = rsv_df[
    ["date", "metric_value"]
].copy()

rsv_clean = rsv_clean.rename(
    columns={"metric_value": "positivity"}
)

rsv_clean["date"] = pd.to_datetime(
    rsv_clean["date"],
    errors="coerce"
)

rsv_clean["positivity"] = pd.to_numeric(
    rsv_clean["positivity"],
    errors="coerce"
)

rsv_clean = rsv_clean.dropna(
    subset=["date", "positivity"]
)

rsv_clean = rsv_clean.sort_values("date")

# Check that each week has one observation
assert not rsv_clean["date"].duplicated().any(), (
    "Multiple observations remain for the same date."
)

# Check positivity is a percentage
assert rsv_clean["positivity"].between(0, 100).all()

output_path = Path("data/processed/rsv_clean.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

rsv_clean.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(rsv_clean))

display(rsv_clean.head())

Saved: data/processed/rsv_clean.csv
Rows: 483


,date,positivity
5,2017-07-03,0.29
7,2017-07-10,0.46
20,2017-07-17,0.40
26,2017-07-24,0.81
34,2017-07-31,1.17


In [ ]:
Clean the data

In [4]:
from pathlib import Path

rsv_clean = rsv_df[
    ["date", "metric_value"]
].copy()

rsv_clean = rsv_clean.rename(
    columns={"metric_value": "positivity"}
)

rsv_clean["date"] = pd.to_datetime(
    rsv_clean["date"],
    errors="coerce"
)

rsv_clean["positivity"] = pd.to_numeric(
    rsv_clean["positivity"],
    errors="coerce"
)

rsv_clean = rsv_clean.dropna(
    subset=["date", "positivity"]
)

rsv_clean = rsv_clean.sort_values("date")

# Check that each week has one observation
assert not rsv_clean["date"].duplicated().any(), (
    "Multiple observations remain for the same date."
)

# Check positivity is a percentage
assert rsv_clean["positivity"].between(0, 100).all()

output_path = Path("data/processed/rsv_clean.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

rsv_clean.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(rsv_clean))

display(rsv_clean.head())

Saved: data/processed/rsv_clean.csv
Rows: 483


,date,positivity
5,2017-07-03,0.29
7,2017-07-10,0.46
20,2017-07-17,0.40
26,2017-07-24,0.81
34,2017-07-31,1.17


In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

# Convert downloaded JSON records into a DataFrame
df = pd.DataFrame(all_results)

print("Original dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())

display(df.head())
print("Available age groups:", df["age"].unique())
print("Available sex categories:", df["sex"].unique())
print("Available strata:", df["stratum"].unique())

print("\nMissing values:")
print(df[["date", "metric_value"]].isna().sum())

print("\nDate range:")
print(df["date"].min(), "to", df["date"].max())
# Make a copy so the original dataset stays unchanged
rsv = df.copy()

# Filter to the population we want
rsv = rsv[
    (rsv["geography"] == "England") &
    (rsv["age"] == "all") &
    (rsv["sex"] == "all") &
    (rsv["stratum"] == "default")
].copy()

print("Records after filtering:", len(rsv))

display(rsv.head())
# Keep relevant columns
rsv_clean = rsv[
    ["date", "metric_value"]
].copy()

# Rename the metric column
rsv_clean = rsv_clean.rename(
    columns={"metric_value": "positivity"}
)

# Convert dates into datetime format
rsv_clean["date"] = pd.to_datetime(
    rsv_clean["date"],
    errors="coerce"
)

# Convert positivity values into numbers
rsv_clean["positivity"] = pd.to_numeric(
    rsv_clean["positivity"],
    errors="coerce"
)

# Remove rows with missing or invalid values
rsv_clean = rsv_clean.dropna(
    subset=["date", "positivity"]
)

# Keep valid percentages only
rsv_clean = rsv_clean[
    rsv_clean["positivity"].between(0, 100)
].copy()

# Sort observations chronologically
rsv_clean = rsv_clean.sort_values("date")

print("Cleaned dataset shape:", rsv_clean.shape)

display(rsv_clean.head())
duplicates = rsv_clean[
    rsv_clean.duplicated(subset=["date"], keep=False)
]

print("Duplicate date rows:", len(duplicates))

display(duplicates.head(10))
# Identify the starting year of each season
rsv_clean["season_start"] = np.where(
    rsv_clean["date"].dt.month >= 7,
    rsv_clean["date"].dt.year,
    rsv_clean["date"].dt.year - 1
)

# Create season labels
rsv_clean["season"] = (
    rsv_clean["season_start"].astype(str)
    + "-"
    + (rsv_clean["season_start"] + 1).astype(str)
)

# Calculate the start date of each season
season_start_dates = pd.to_datetime(
    rsv_clean["season_start"].astype(str) + "-07-01"
)

# Calculate the week within each season
rsv_clean["season_week"] = (
    (rsv_clean["date"] - season_start_dates).dt.days // 7
) + 1

rsv_clean = rsv_clean.reset_index(drop=True)

display(rsv_clean.head())
output_path = Path(
    "/home/jovyan/RSVision_Hackathon/Hackathon/data/processed/rsv_clean.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

rsv_clean.to_csv(
    output_path,
    index=False
)

print("Cleaned RSV dataset saved successfully!")
print("Location:", output_path)
print("Total observations:", len(rsv_clean))
print("Final dataset:", rsv_clean.shape)

print("\nMissing values:")
print(rsv_clean.isna().sum())

print("\nDuplicate dates:")
print(rsv_clean["date"].duplicated().sum())

print("\nDate range:")
print(
    rsv_clean["date"].min(),
    "to",
    rsv_clean["date"].max()
)

print("\nSeasons:")
print(rsv_clean["season"].value_counts().sort_index())

print("\nPositivity summary:")
display(rsv_clean["positivity"].describe())

Original dataset shape: (3532, 17)

Columns: ['theme', 'sub_theme', 'topic', 'geography_type', 'geography', 'geography_code', 'metric', 'metric_group', 'stratum', 'sex', 'age', 'year', 'month', 'epiweek', 'date', 'metric_value', 'in_reporting_delay_period']


,theme,sub_theme,topic,geography_type,geography,geography_code,metric,metric_group,stratum,sex,age,year,month,epiweek,date,metric_value,in_reporting_delay_period
0,infectious_disease,respiratory,RSV,Nation,England,E92000001,RSV_testing_positivityByWeek,testing,default,all,45-64,2017,7,27,2017-07-03,0.00,False
1,infectious_disease,respiratory,RSV,Nation,England,E92000001,RSV_testing_positivityByWeek,testing,default,all,80+,2017,7,27,2017-07-03,0.00,False
2,infectious_disease,respiratory,RSV,Nation,England,E92000001,RSV_testing_positivityByWeek,testing,default,all,00-04,2017,7,27,2017-07-03,0.69,False
3,infectious_disease,respiratory,RSV,Nation,England,E92000001,RSV_testing_positivityByWeek,testing,default,all,65-79,2017,7,27,2017-07-03,0.59,False
4,infectious_disease,respiratory,RSV,Nation,England,E92000001,RSV_testing_positivityByWeek,testing,default,all,05-14,2017,7,27,2017-07-03,0.00,False


Available age groups: ['45-64' '80+' '00-04' '65-79' '05-14' 'all' '15-44' '65+' '65']
Available sex categories: ['all']
Available strata: ['default']

Missing values:
date            0
metric_value    0
dtype: int64

Date range:
2017-07-03 to 2026-09-28
Records after filtering: 483


,theme,sub_theme,topic,geography_type,geography,geography_code,metric,metric_group,stratum,sex,age,year,month,epiweek,date,metric_value,in_reporting_delay_period
5,infectious_disease,respiratory,RSV,Nation,England,E92000001,RSV_testing_positivityByWeek,testing,default,all,all,2017,7,27,2017-07-03,0.29,False
7,infectious_disease,respiratory,RSV,Nation,England,E92000001,RSV_testing_positivityByWeek,testing,default,all,all,2017,7,28,2017-07-10,0.46,False
20,infectious_disease,respiratory,RSV,Nation,England,E92000001,RSV_testing_positivityByWeek,testing,default,all,all,2017,7,29,2017-07-17,0.40,False
26,infectious_disease,respiratory,RSV,Nation,England,E92000001,RSV_testing_positivityByWeek,testing,default,all,all,2017,7,30,2017-07-24,0.81,False
34,infectious_disease,respiratory,RSV,Nation,England,E92000001,RSV_testing_positivityByWeek,testing,default,all,all,2017,7,31,2017-07-31,1.17,False


Cleaned dataset shape: (483, 2)


...

Duplicate date rows: 0


,date,positivity


,date,positivity,season_start,season,season_week
0,2017-07-03,0.29,2017,2017-2018,1
1,2017-07-10,0.46,2017,2017-2018,2
2,2017-07-17,0.40,2017,2017-2018,3
3,2017-07-24,0.81,2017,2017-2018,4
4,2017-07-31,1.17,2017,2017-2018,5


Cleaned RSV dataset saved successfully!
Location: /home/jovyan/RSVision_Hackathon/Hackathon/data/processed/rsv_clean.csv
Total observations: 483
Final dataset: (483, 5)

Missing values:
date            0
positivity      0
season_start    0
season          0
season_week     0
dtype: int64

Duplicate dates:
0

Date range:
2017-07-03 00:00:00 to 2026-09-28 00:00:00

Seasons:
season
2017-2018    52
2018-2019    52
2019-2020    53
2020-2021    52
2021-2022    52
2022-2023    52
2023-2024    52
2024-2025    53
2025-2026    52
2026-2027    13
Name: count, dtype: int64

Positivity summary:


count    483.000000
mean       3.513126
std        4.837288
min        0.000000
25%        0.400000
50%        1.240000
75%        4.685000
max       25.350000
Name: positivity, dtype: float64